# MVP - Data Engineering

In [0]:
# 1. Python Libraries

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [0]:
%sql
-- Estabelecendo o catálogo e schema que serão usados no MVP (SQL)

USE CATALOG mvp_pucrio;
USE SCHEMA gold;

### Análise

#### Quais são os meses de maior pico?

In [0]:
# Quais são os meses de maior pico?

total_sales_monthly = spark.sql("""
    SELECT
        t.month, t.year,
        COUNT(DISTINCT f.order_id) AS num_orders,
        ROUND(SUM(f.sales_value), 2) AS total_sales_value
    FROM dim_dates t
    INNER JOIN fato_vendas f
        ON t.order_date = f.order_date
    GROUP BY t.month, t.year
    ORDER BY t.year, t.month
""")

df_monthly = total_sales_monthly.toPandas()

df_monthly['periodo'] = df_monthly['year'].astype(str) + '-' + df_monthly['month'].astype(str).str.zfill(2)
df_monthly = df_monthly.sort_values(['year', 'month']).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(df_monthly['periodo'], df_monthly['total_sales_value'])

ax.set_xlabel('Mes/Ano', fontsize=11)
ax.set_ylabel('Valor vendido (R$)', fontsize=11)
ax.set_title('Valor vendido por mês (R$)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'R$ {x/1000:.0f}K'))
plt.xticks(rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.show()

Como vemos acima, o total vendido vem aumentando gradativamente desde outubro/2016 até meados de 2018. Vemos um pico de vendas em novembro/2017 (R$ 1M), provavelmente ocasionado pela Black Friday.

**Ponto de atenção:** Entre julho e setembro/2018, as vendas caem de aproximadamente R$ 1M para R$ 850K (-15%). Isso pode indicar sazonalidade natural ou perda de tração da plataforma. Seria importante investigar se essa queda se mantém nos meses seguintes.

Precisaríamos de um período maior (3 a 5 anos de 2018 em diante) para entender efetivamente se há picos de venda recorrentes.

Para as **próximas análises**, vamos focar no período de setembro/2017 a agosto/2018, para ter um ano completo. Considerar o período anterior não traria uma boa análise já que parece ser o período de início destes vendedores na plataforma, antes das vendas se estabilizarem (a não ser que se quisesse observar algo específico neste período inicial).

**Recomendação:** Reforçar ações de retenção e reativação de sellers no período pós-Black Friday (dez–fev), historicamente mais fraco; investigar se a queda em 2018 persiste nos meses seguintes para distinguir sazonalidade de perda de tração.

#### De onde vem a receita, i.e. a regra de Pareto se aplica aos vendedores da plataforma?

In [0]:
%sql
-- De onde vem a receita, i.e. a regra de Pareto se aplica aos vendedores da plataforma?

WITH sales_per_seller AS (
    SELECT
        seller_id,
        COUNT(DISTINCT f.order_id) AS num_orders,
        ROUND(SUM(f.sales_value), 2) AS total_sales,
        ROUND(SUM(f.commission), 2) AS total_commission,
        ROUND(SUM(f.sales_value) / COUNT(DISTINCT f.order_id), 2) AS avg_ticket
    FROM fato_vendas f
    WHERE f.order_date >= '2017-09-01'
    GROUP BY seller_id
)
SELECT 
    seller_id,
    num_orders,
    total_sales,
    total_commission,
    avg_ticket,
    ROUND(total_sales * 100 / SUM(total_sales) OVER (), 2) AS percentage_of_total_sales,
    ROUND(
        SUM(total_sales) 
            OVER (ORDER BY total_sales DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) * 100 / 
        SUM(total_sales) OVER (), 2
    ) AS cumm_percentage_of_total_sales
FROM sales_per_seller
ORDER BY total_sales DESC;

Observando os vendedores com vendas no período (a partir de setembro/2017), 512 dos 2.653 vendedores (19,3%) detêm aproximadamente 80% das vendas, confirmando a regra 80/20 do princípio de Pareto. Apenas 122 sellers (aproximadamente 5%) acumulam 50% do faturamento.

Entre os top 10, o ticket médio varia de R$ 67 a R$ 581, revelando dois modelos de negócio distintos entre os maiores sellers. Como a comissão é fixa em 10%, os top sellers também são os que mais geram receita para a plataforma (top seller: R$ 20,4K de comissão no período).

Os 81% restantes (2.141 sellers) dividem apenas 20% das vendas, configurando uma longa cauda de vendedores de baixo volume.

**Recomendação:** Apesar de confirmar a regra de Pareto, a receita está bem distribuída dentro do grupo dos 512 vendedores que mais vendem (o maior responde por apenas 2% do total), sendo aconselhável acompanhar de perto o desempenho deles e agir rápido se algum começar a vender menos. Já para o restante, é recomendável uma análise mais profunda para entender se têm poucos produtos cadastrados, se o preço não é competitivo, se falta divulgação, ou outro motivo e, com base nisso, definir o plano de ação.

In [0]:
%sql
-- Sumário: top sellers (19% que detêm 80%) vs restante (81% que detêm 20%)

WITH sales_per_seller AS (
    SELECT
        seller_id,
        COUNT(DISTINCT f.order_id) AS num_orders,
        ROUND(SUM(f.sales_value), 2) AS total_sales,
        ROUND(SUM(f.commission), 2) AS total_commission,
        ROUND(SUM(f.sales_value) / COUNT(DISTINCT f.order_id), 2) AS avg_ticket
    FROM fato_vendas f
    WHERE f.order_date >= '2017-09-01'
    GROUP BY seller_id
),
ranked AS (
    SELECT 
        seller_id,
        num_orders,
        total_sales,
        total_commission,
        avg_ticket,
        ROUND(SUM(total_sales) OVER (ORDER BY total_sales DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) * 100 / SUM(total_sales) OVER (), 2) AS cum_pct
    FROM sales_per_seller
)
SELECT 
    CASE WHEN cum_pct <= 80 THEN 'Top 19%' ELSE 'Restante 81%' END AS grupo,
    COUNT(DISTINCT seller_id) AS num_sellers,
    ROUND(COUNT(DISTINCT seller_id) * 100.0 / SUM(COUNT(DISTINCT seller_id)) OVER (), 1) AS pct_sellers,
    SUM(num_orders) AS total_orders,
    ROUND(SUM(total_sales), 2) AS total_sales,
    ROUND(SUM(total_sales) * 100 / SUM(SUM(total_sales)) OVER (), 1) AS pct_sales,
    ROUND(SUM(total_commission), 2) AS total_commission,
    ROUND(AVG(avg_ticket), 2) AS avg_ticket
FROM ranked
GROUP BY CASE WHEN cum_pct <= 80 THEN 'Top 19%' ELSE 'Restante 81%' END
ORDER BY total_sales DESC;


#### Quais segmentos de produto trazem mais receita?

In [0]:
%sql
-- Quais segmentos de produto trazem mais receita?

SELECT 
    p.category,
    COUNT(DISTINCT f.order_id) AS num_orders,
    ROUND(SUM(f.sales_value), 2) AS total_sales,
    ROUND(AVG(f.sales_value), 2) AS avg_ticket,
    ROUND(SUM(f.commission), 2) AS total_commission
FROM fato_vendas f
LEFT JOIN dim_products p
    ON f.product_id = p.product_id
WHERE order_date >= '2017-09-01'
GROUP BY p.category
ORDER BY total_sales DESC
LIMIT 20;

A categoria beleza_saude lidera as vendas (R$ 1M, 7.056 pedidos), seguida por relogios_presentes (R$ 991K, 4.811 pedidos). As categorias cama_mesa_banho, esporte_lazer e informática completam o top 5, confirmando que utilidade doméstica, bem-estar e lazer dominam o catálogo mais vendido.

Essas top 5 representam aproximadamente 43% do valor total vendido, indicando que não há dependência crítica de uma única categoria.

A categoria PCs (17ª posição) tem o maior ticket médio da plataforma (R$ 1.223), mas apenas 150 pedidos. Investir em marketing para essa categoria poderia ajudar a aumentar volume. 

#### Algum canal de marketing é melhor (em termos de conversão e ciclo médio de vendas)?

In [0]:
%sql
-- Algum canal de marketing é melhor (em termos de conversão e ciclo médio de vendas)?

SELECT
    origin,
    COUNT(mql_id) AS leads_count,
    COUNT(won_date) AS converted_leads_count,
    ROUND((COUNT(won_date) / COUNT(mql_id)) * 100, 2) AS conversion_rate_percentage,
    ROUND(AVG(sales_cycle), 1) AS avg_sales_cycle_days,
    ROUND(MIN(sales_cycle), 1) AS min_cycle_days,
    ROUND(MAX(sales_cycle), 1) AS max_cycle_days
FROM dim_leads
GROUP BY origin
HAVING 
    origin != "unknown" AND
    origin != "other" AND
    origin != "other_publicities"
ORDER BY conversion_rate_percentage DESC;

Os três canais com maior taxa de conversão são pesquisas pagas (12,3%), buscas orgânicas (11,8%) e tráfego direto (11,2%), sendo que os dois primeiros também lideram em número de leads. Já social, display e email têm conversão bem mais baixa (5,6%, 5,1% e 3,0%).

No entanto, podemos notar que apesar da conversão baixa, social tem um volume maior, trazendo mais leads convertidos (75) do que referral (24), que tem taxa de conversão um pouco maior.

Podemos notar também que display tem o ciclo de vendas mais curto (10,3 dias, contra 50-60 dias de paid_search/organic_search), mas sua baixa conversão (5,1%) sugere que a rapidez não compensa o volume perdido.

**Recomendações:**
* **Priorizar investimento em paid_search e organic_search** — melhor ROI (maior conversão + alto volume). Paid_search já traz 195 conversões; aumentar investimento pode escalar ainda mais.
* **Otimizar social** — tem volume alto (1.350 leads) mas conversão metade da média (5,6%). Melhorar qualidade dos leads ou jornada pós-clique pode dobrar as conversões sem aumentar custo de aquisição.
* **Reavaliar display** — ciclo curto (10 dias) é atrativo, mas conversão de 5,1% e apenas 6 conversões totais sugerem baixo retorno. Considerar realocar orçamento para canais de maior conversão.

#### Existe relação entre baixo faturamento e maior risco de abandonar a plataforma? (LTV, churn)

In [0]:
%sql
-- Existe relação entre baixo faturamento e maior risco de abandonar a plataforma? (LTV, churn)

WITH seller_metrics AS (
    SELECT 
        f.seller_id,
        COUNT(DISTINCT f.order_id) AS num_orders,
        ROUND(SUM(f.commission), 2) AS total_commission,
        MIN(f.order_date) AS first_sale_date,
        MAX(f.order_date) AS last_sale_date,
        DATEDIFF('2018-08-31', MAX(f.order_date)) AS days_since_last_sale,
        CASE 
            WHEN DATEDIFF('2018-08-31', MAX(f.order_date)) > 180 THEN 'Churned'
            ELSE 'Active'
        END AS seller_status
    FROM fato_vendas f
    WHERE f.order_date >= '2017-09-01'
    GROUP BY f.seller_id
)
SELECT 
    seller_status,
    COUNT(DISTINCT seller_id) AS num_sellers,
    ROUND((COUNT(DISTINCT seller_id) * 100.0 / SUM(COUNT(DISTINCT seller_id)) OVER ()), 2) AS percentage_sellers,
    ROUND(AVG(total_commission), 2) AS avg_ltv_commission,
    ROUND(AVG(num_orders), 2) AS avg_num_orders_per_seller,
    ROUND(AVG(days_since_last_sale), 1) AS avg_days_since_last_sale
FROM seller_metrics
GROUP BY seller_status
ORDER BY seller_status;

Dos 2.653 vendedores da plataforma, 2.166 (81,6%) estão ativos e 487 (18,4%) estão churned (sem vendas há mais de 180 dias, considerando 31/08/2018 como referência).

O LTV médio de comissão de um vendedor ativo é R$ 462, contra R$ 73 de um churned (6,3x maior). Vendedores ativos também vendem muito mais (34 pedidos em média vs 4). 

Isso indica que o churn está associado a baixo engajamento, ou seja, vendedores que não conseguiram crescer as vendas tendem a abandonar a plataforma. Isso sugere que a implementação de ações que ajudem o vendedor a ter uma boa experiência nas primeiras vendas poderiam impactar a retenção.

**Recomendação:** Como o abandono está ligado a vendedores que não conseguem vender bem logo no início, faria sentido identificar automaticamente quem fez a primeira venda mas não fez uma segunda depois de um tempo (por exemplo, 90 dias) e entrar em contato oferecendo ajuda.